INSTALL WORLD BANK API



In [ ]:
!pip install wbgapi


In [ ]:
import wbgapi as wb
import pandas as pd
import numpy as np

In [ ]:
# Define which indicators we want and give them readable names
indicators = {
    'NY.GDP.MKTP.CD': 'GDP_USD',
    'FP.CPI.TOTL.ZG': 'Inflation_pct',
    'PA.NUS.FCRF': 'Exchange_rate'
}

# Fetch data for Nigeria (country code NGA) from 2000 to 2024
df = wb.data.DataFrame(
    list(indicators.keys()),
    economy='NGA',
    time=range(2000, 2025)
)

# Rename columns to our readable names
df = df.rename(columns=indicators)

# Show the first few rows
print(df.head(10))

                      YR2000        YR2001        YR2002        YR2003  \
series                                                                   
FP.CPI.TOTL.ZG  6.933292e+00  1.887365e+01  1.287658e+01  1.403178e+01   
NY.GDP.MKTP.CD  6.917145e+10  7.355784e+10  9.505406e+10  1.047390e+11   
PA.NUS.FCRF     1.016973e+02  1.112313e+02  1.205782e+02  1.292224e+02   

                      YR2004        YR2005        YR2006        YR2007  \
series                                                                   
FP.CPI.TOTL.ZG  1.499803e+01  1.786349e+01  8.227437e+00  5.386104e+00   
NY.GDP.MKTP.CD  1.357647e+11  1.756706e+11  2.384550e+11  2.782608e+11   
PA.NUS.FCRF     1.328880e+02  1.312743e+02  1.286517e+02  1.258081e+02   

                      YR2008        YR2009  ...        YR2015        YR2016  \
series                                      ...                               
FP.CPI.TOTL.ZG  1.158305e+01  1.253792e+01  ...  9.009435e+00  1.569681e+01   
NY.GDP.MKTP.CD  3.394

In [ ]:
# Show every row that has at least one missing value
print(df[df.isnull().any(axis=1)])

Empty DataFrame
Columns: [YR2000, YR2001, YR2002, YR2003, YR2004, YR2005, YR2006, YR2007, YR2008, YR2009, YR2010, YR2011, YR2012, YR2013, YR2014, YR2015, YR2016, YR2017, YR2018, YR2019, YR2020, YR2021, YR2022, YR2023, YR2024]
Index: []

[0 rows x 25 columns]


In [ ]:
# Show the last 5 rows so we can see the most recent data
print(df.tail(5))

                      YR2000        YR2001        YR2002        YR2003  \
series                                                                   
FP.CPI.TOTL.ZG  6.933292e+00  1.887365e+01  1.287658e+01  1.403178e+01   
NY.GDP.MKTP.CD  6.917145e+10  7.355784e+10  9.505406e+10  1.047390e+11   
PA.NUS.FCRF     1.016973e+02  1.112313e+02  1.205782e+02  1.292224e+02   

                      YR2004        YR2005        YR2006        YR2007  \
series                                                                   
FP.CPI.TOTL.ZG  1.499803e+01  1.786349e+01  8.227437e+00  5.386104e+00   
NY.GDP.MKTP.CD  1.357647e+11  1.756706e+11  2.384550e+11  2.782608e+11   
PA.NUS.FCRF     1.328880e+02  1.312743e+02  1.286517e+02  1.258081e+02   

                      YR2008        YR2009  ...        YR2015        YR2016  \
series                                      ...                               
FP.CPI.TOTL.ZG  1.158305e+01  1.253792e+01  ...  9.009435e+00  1.569681e+01   
NY.GDP.MKTP.CD  3.394

In [ ]:
# Step 1 — Transpose so years become rows
df = df.T

# Step 2 — Clean the year index
df.index = df.index.str.replace('YR', '').astype(int)
df.index.name = 'Year'

# Step 3 — Rename columns to readable names
df.columns = ['Inflation_pct', 'GDP_USD', 'Exchange_rate']

# Step 4 — Convert GDP to billions and drop the raw column
df['GDP_Billions'] = (df['GDP_USD'] / 1e9).round(2)
df = df.drop(columns=['GDP_USD'])

# Step 5 — Round everything
df = df.round(2)

# Step 6 — Save a clean copy
df.to_csv('nigeria_clean.csv')

# Check the result
print(df)

      Inflation_pct  Exchange_rate  GDP_Billions
Year                                            
2000           6.93         101.70         69.17
2001          18.87         111.23         73.56
2002          12.88         120.58         95.05
2003          14.03         129.22        104.74
2004          15.00         132.89        135.76
2005          17.86         131.27        175.67
2006           8.23         128.65        238.45
2007           5.39         125.81        278.26
2008          11.58         118.57        339.48
2009          12.54         148.88        295.01
2010          13.74         150.30        366.99
2011          10.83         153.86        414.47
2012          12.22         157.50        463.97
2013           8.50         157.31        520.12
2014           8.05         158.55        574.18
2015           9.01         192.44        493.03
2016          15.70         253.49        404.65
2017          16.50         305.79        375.75
2018          12.10 

CALCULATE YEAR-ON-YEAR PERCENTAGE CHANGE

In [ ]:
# Year-on-year change for GDP
df['GDP_YoY_pct'] = df['GDP_Billions'].pct_change() * 100

# Year-on-year change for exchange rate
df['FX_YoY_pct'] = df['Exchange_rate'].pct_change() * 100

df = df.round(2)
print(df[['GDP_Billions', 'GDP_YoY_pct', 'Exchange_rate', 'FX_YoY_pct']])

      GDP_Billions  GDP_YoY_pct  Exchange_rate  FX_YoY_pct
Year                                                      
2000         69.17          NaN         101.70         NaN
2001         73.56         6.35         111.23        9.37
2002         95.05        29.21         120.58        8.41
2003        104.74        10.19         129.22        7.17
2004        135.76        29.62         132.89        2.84
2005        175.67        29.40         131.27       -1.22
2006        238.45        35.74         128.65       -2.00
2007        278.26        16.70         125.81       -2.21
2008        339.48        22.00         118.57       -5.75
2009        295.01       -13.10         148.88       25.56
2010        366.99        24.40         150.30        0.95
2011        414.47        12.94         153.86        2.37
2012        463.97        11.94         157.50        2.37
2013        520.12        12.10         157.31       -0.12
2014        574.18        10.39         158.55        0.

CALCULATE ROLLING AVERAGES

In [ ]:
# 3-year rolling average for inflation
df['Inflation_3yr_avg'] = df['Inflation_pct'].rolling(window=3).mean()

# 3-year rolling average for GDP growth
df['GDP_3yr_avg'] = df['GDP_YoY_pct'].rolling(window=3).mean()

df = df.round(2)
print(df[['Inflation_pct', 'Inflation_3yr_avg', 'GDP_YoY_pct', 'GDP_3yr_avg']])

      Inflation_pct  Inflation_3yr_avg  GDP_YoY_pct  GDP_3yr_avg
Year                                                            
2000           6.93                NaN          NaN          NaN
2001          18.87                NaN         6.35          NaN
2002          12.88              12.89        29.21          NaN
2003          14.03              15.26        10.19        15.25
2004          15.00              13.97        29.62        23.01
2005          17.86              15.63        29.40        23.07
2006           8.23              13.70        35.74        31.59
2007           5.39              10.49        16.70        27.28
2008          11.58               8.40        22.00        24.81
2009          12.54               9.84       -13.10         8.53
2010          13.74              12.62        24.40        11.10
2011          10.83              12.37        12.94         8.08
2012          12.22              12.26        11.94        16.43
2013           8.50      

EXTREMES IN THE DATASET


In [ ]:
print("Highest inflation year:", df['Inflation_pct'].idxmax(), "-", df['Inflation_pct'].max(), "%")
print("Lowest inflation year:", df['Inflation_pct'].idxmin(), "-", df['Inflation_pct'].min(), "%")
print("Highest GDP year:", df['GDP_Billions'].idxmax(), "-", df['GDP_Billions'].max(), "billion USD")
print("Largest single-year FX devaluation:", df['FX_YoY_pct'].idxmax(), "-", df['FX_YoY_pct'].max(), "%")

Highest inflation year: 2024 - 33.24 %
Lowest inflation year: 2007 - 5.39 %
Highest GDP year: 2019 - 668.22 billion USD
Largest single-year FX devaluation: 2024 - 129.23 %


In [ ]:
!pip install wbgapi scikit-learn plotly pandas numpy -q

import wbgapi as wb
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, LeaveOneOut
from sklearn.metrics import mean_absolute_error, r2_score
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("All libraries loaded successfully")

All libraries loaded successfully


In [ ]:
indicators = {
    'NY.GDP.MKTP.CD': 'GDP_USD',
    'FP.CPI.TOTL.ZG': 'Inflation_pct',
    'PA.NUS.FCRF': 'Exchange_rate',
    'NE.GDI.TOTL.ZS': 'Investment_pct_GDP',
    'NE.IMP.GNFS.ZS': 'Imports_pct_GDP',
    'SL.UEM.TOTL.ZS': 'Unemployment_pct',
}

df = wb.data.DataFrame(
    list(indicators.keys()),
    economy='NGA',
    time=range(2000, 2025)
)

# Clean and reshape
df = df.T
df = df.rename(columns=indicators) # Renaming columns after transposing
df.index = df.index.str.replace('YR', '').astype(int)
df.index.name = 'Year'
df['GDP_Billions'] = (df['GDP_USD'] / 1e9).round(2)
df = df.drop(columns=['GDP_USD'])
df = df.sort_values('Year').round(2)

print(df.shape)
print(df.isnull().sum())
print(df)

(25, 6)
series
Inflation_pct          0
Investment_pct_GDP    25
Imports_pct_GDP       25
Exchange_rate          0
Unemployment_pct       0
GDP_Billions           0
dtype: int64
series  Inflation_pct  Investment_pct_GDP  Imports_pct_GDP  Exchange_rate  \
Year                                                                        
2000             6.93                 NaN              NaN         101.70   
2001            18.87                 NaN              NaN         111.23   
2002            12.88                 NaN              NaN         120.58   
2003            14.03                 NaN              NaN         129.22   
2004            15.00                 NaN              NaN         132.89   
2005            17.86                 NaN              NaN         131.27   
2006             8.23                 NaN              NaN         128.65   
2007             5.39                 NaN              NaN         125.81   
2008            11.58                 NaN           

REPLACE MISSING COLUMNS WITH BETTER ALTERNATIVES

In [ ]:
# Drop the two completely empty columns
df = df.drop(columns=['Investment_pct_GDP', 'Imports_pct_GDP'])

# Add oil prices manually — this is more reliable and more relevant
oil_prices = {
    2000: 28.5, 2001: 24.4, 2002: 25.0, 2003: 28.8, 2004: 38.3,
    2005: 54.5, 2006: 65.1, 2007: 72.4, 2008: 99.0, 2009: 61.5,
    2010: 79.5, 2011: 111.3, 2012: 111.6, 2013: 108.6, 2014: 98.9,
    2015: 52.4, 2016: 43.6, 2017: 54.2, 2018: 71.1, 2019: 64.4,
    2020: 41.8, 2021: 70.9, 2022: 100.9, 2023: 82.2, 2024: 79.8
}
df['Oil_price'] = pd.Series(oil_prices)

# Add policy shock flag — years of major structural breaks
df['Policy_shock'] = df.index.isin([2016, 2024]).astype(int)

# Add GDP in constant prices to remove exchange rate distortion
gdp_constant = {
    2000: 100.0, 2001: 104.7, 2002: 115.9, 2003: 123.1, 2004: 136.1,
    2005: 148.1, 2006: 160.2, 2007: 174.9, 2008: 188.3, 2009: 196.4,
    2010: 212.8, 2011: 229.7, 2012: 246.2, 2013: 264.5, 2014: 282.4,
    2015: 289.3, 2016: 286.7, 2017: 291.2, 2018: 302.7, 2019: 314.5,
    2020: 306.5, 2021: 323.9, 2022: 340.2, 2023: 354.8, 2024: 361.2
}
df['GDP_Constant'] = pd.Series(gdp_constant)

print(df.shape)
print(df.isnull().sum())
print(df.tail(5))

(25, 7)
series
Inflation_pct       0
Exchange_rate       0
Unemployment_pct    0
GDP_Billions        0
Oil_price           0
Policy_shock        0
GDP_Constant        0
dtype: int64
series  Inflation_pct  Exchange_rate  Unemployment_pct  GDP_Billions  \
Year                                                                   
2020            13.25         358.81              5.71        598.59   
2021            16.95         401.15              5.40        609.15   
2022            18.85         425.98              3.83        646.95   
2023            24.66         645.19              3.07        487.39   
2024            33.24        1478.97              3.04        252.26   

series  Oil_price  Policy_shock  GDP_Constant  
Year                                           
2020         41.8             0         306.5  
2021         70.9             0         323.9  
2022        100.9             0         340.2  
2023         82.2             0         354.8  
2024         79.8        

FEATURE ENGINEERING - CREATING LAG FEATURES

In [ ]:
# Create a copy to work on
df_model = df.copy()

# Lag features — previous year's values
lag_cols = ['Inflation_pct', 'Exchange_rate', 'GDP_Constant', 'Oil_price']
for col in lag_cols:
    df_model[f'{col}_lag1'] = df_model[col].shift(1)

print("After adding lag features:")
print(df_model.shape)
print(df_model.head(3))

After adding lag features:
(25, 11)
series  Inflation_pct  Exchange_rate  Unemployment_pct  GDP_Billions  \
Year                                                                   
2000             6.93         101.70              3.95         69.17   
2001            18.87         111.23              3.90         73.56   
2002            12.88         120.58              3.67         95.05   

series  Oil_price  Policy_shock  GDP_Constant  Inflation_pct_lag1  \
Year                                                                
2000         28.5             0         100.0                 NaN   
2001         24.4             0         104.7                6.93   
2002         25.0             0         115.9               18.87   

series  Exchange_rate_lag1  GDP_Constant_lag1  Oil_price_lag1  
Year                                                           
2000                   NaN                NaN             NaN  
2001                101.70              100.0            28.5  
2

CREATING RATE OF CHANGE FEATURES

In [ ]:
# Year-on-year percentage changes
df_model['Inflation_YoY'] = df_model['Inflation_pct'].pct_change() * 100
df_model['Exchange_YoY'] = df_model['Exchange_rate'].pct_change() * 100
df_model['GDP_YoY'] = df_model['GDP_Constant'].pct_change() * 100
df_model['Oil_YoY'] = df_model['Oil_price'].pct_change() * 100

print("After adding rate of change features:")
print(df_model.shape)

After adding rate of change features:
(25, 15)


CREATING INTERACTIVE FEATURES

In [ ]:
# Oil revenue proxy — oil price × GDP size
df_model['Oil_GDP_interaction'] = df_model['Oil_price'] * df_model['GDP_Constant'] / 100

# Inflation pressure — how much inflation is accelerating
df_model['Inflation_acceleration'] = df_model['Inflation_pct'] - df_model['Inflation_pct'].shift(2)

print("After adding interaction features:")
print(df_model.shape)

After adding interaction features:
(25, 17)


DROP THE FIRST TWO ROWS AND REST

In [ ]:
# Drop rows with NaN from lag creation
df_model = df_model.dropna().reset_index()

print(f"Final model dataset: {df_model.shape}")
print(df_model.columns.tolist())
print(df_model.tail(3))

Final model dataset: (23, 18)
['Year', 'Inflation_pct', 'Exchange_rate', 'Unemployment_pct', 'GDP_Billions', 'Oil_price', 'Policy_shock', 'GDP_Constant', 'Inflation_pct_lag1', 'Exchange_rate_lag1', 'GDP_Constant_lag1', 'Oil_price_lag1', 'Inflation_YoY', 'Exchange_YoY', 'GDP_YoY', 'Oil_YoY', 'Oil_GDP_interaction', 'Inflation_acceleration']
series  Year  Inflation_pct  Exchange_rate  Unemployment_pct  GDP_Billions  \
20      2022          18.85         425.98              3.83        646.95   
21      2023          24.66         645.19              3.07        487.39   
22      2024          33.24        1478.97              3.04        252.26   

series  Oil_price  Policy_shock  GDP_Constant  Inflation_pct_lag1  \
20          100.9             0         340.2               16.95   
21           82.2             0         354.8               18.85   
22           79.8             1         361.2               24.66   

series  Exchange_rate_lag1  GDP_Constant_lag1  Oil_price_lag1  Inflat

TRAINING THE MACHINE LANGUAGE MODEL - DEFINING FEATURES AND TARGETS

In [ ]:
# Features — what the model uses as input
feature_cols = [
    'Inflation_pct_lag1',
    'Exchange_rate_lag1',
    'GDP_Constant_lag1',
    'Oil_price_lag1',
    'Unemployment_pct',
    'Policy_shock',
    'Inflation_YoY',
    'Exchange_YoY',
    'GDP_YoY',
    'Oil_YoY',
    'Oil_GDP_interaction',
    'Inflation_acceleration'
]

# Targets — what the model is trying to predict
target_gdp = 'GDP_Constant'
target_inflation = 'Inflation_pct'

X = df_model[feature_cols]
y_gdp = df_model[target_gdp]
y_inflation = df_model[target_inflation]

print("Features shape:", X.shape)
print("GDP target shape:", y_gdp.shape)
print("Inflation target shape:", y_inflation.shape)
print("\nFeature columns:")
for col in feature_cols:
    print(" -", col)

Features shape: (23, 12)
GDP target shape: (23,)
Inflation target shape: (23,)

Feature columns:
 - Inflation_pct_lag1
 - Exchange_rate_lag1
 - GDP_Constant_lag1
 - Oil_price_lag1
 - Unemployment_pct
 - Policy_shock
 - Inflation_YoY
 - Exchange_YoY
 - GDP_YoY
 - Oil_YoY
 - Oil_GDP_interaction
 - Inflation_acceleration


STARTING OVER TO ACCOMODATE NEW INDICATORS

In [ ]:
import wbgapi as wb
import pandas as pd
import numpy as np

indicators = {
    'NY.GDP.MKTP.KD': 'GDP_Constant',
    'NY.GDP.PCAP.KD': 'GDP_per_capita',
    'FP.CPI.TOTL.ZG': 'Inflation_pct',
    'FP.CPI.TOTL.ZG.FD': 'Food_inflation',
    'PA.NUS.FCRF': 'Exchange_rate',
    'SL.UEM.TOTL.ZS': 'Unemployment_pct',
    'NE.IMP.GNFS.ZS': 'Imports_pct_GDP',
}

df = wb.data.DataFrame(
    list(indicators.keys()),
    economy='NGA',
    time=range(2000, 2025)
)

df = df.T
df = df.rename(columns=indicators) # Renaming columns after transposing
df.index = df.index.str.replace('YR', '').astype(int)
df.index.name = 'Year'
df = df.sort_values('Year').round(2)

# Add manual data
oil_prices = {
    2000:28.5,2001:24.4,2002:25.0,2003:28.8,2004:38.3,
    2005:54.5,2006:65.1,2007:72.4,2008:99.0,2009:61.5,
    2010:79.5,2011:111.3,2012:111.6,2013:108.6,2014:98.9,
    2015:52.4,2016:43.6,2017:54.2,2018:71.1,2019:64.4,
    2020:41.8,2021:70.9,2022:100.9,2023:82.2,2024:79.8
}

us_rates = {
    2000:6.24,2001:3.88,2002:1.67,2003:1.13,2004:1.35,
    2005:3.22,2006:5.02,2007:5.02,2008:1.92,2009:0.24,
    2010:0.18,2011:0.10,2012:0.14,2013:0.11,2014:0.09,
    2015:0.13,2016:0.40,2017:1.00,2018:1.83,2019:2.16,
    2020:0.36,2021:0.08,2022:1.68,2023:5.02,2024:5.33
}

population = {
    2000:122.3,2001:125.9,2002:129.6,2003:133.4,2004:137.3,
    2005:141.4,2006:145.5,2007:149.7,2008:154.0,2009:158.4,
    2010:163.0,2011:167.6,2012:172.4,2013:177.3,2014:182.2,
    2015:187.0,2016:191.8,2017:196.6,2018:201.4,2019:206.1,
    2020:210.9,2021:215.7,2022:220.4,2023:225.1,2024:229.8
}

df['Oil_price'] = pd.Series(oil_prices)
df['US_interest_rate'] = pd.Series(us_rates)
df['Population_millions'] = pd.Series(population)
df['Policy_shock'] = df.index.isin([2016, 2024]).astype(int)

# GDP Billions for reference
df['GDP_Billions'] = (df['GDP_Constant'] / 1e9).round(2) if df['GDP_Constant'].max() > 1e9 else df['GDP_Constant']

print(df.shape)
print(df.isnull().sum())
print(df.tail(3))

(25, 11)
series
Inflation_pct           0
Imports_pct_GDP        25
GDP_Constant            0
GDP_per_capita          0
Exchange_rate           0
Unemployment_pct        0
Oil_price               0
US_interest_rate        0
Population_millions     0
Policy_shock            0
GDP_Billions            0
dtype: int64
series  Inflation_pct  Imports_pct_GDP  GDP_Constant  GDP_per_capita  \
Year                                                                   
2022            18.85              NaN  5.030470e+11         2254.29   
2023            24.66              NaN  5.197276e+11         2280.68   
2024            33.24              NaN  5.408860e+11         2324.60   

series  Exchange_rate  Unemployment_pct  Oil_price  US_interest_rate  \
Year                                                                   
2022           425.98              3.83      100.9              1.68   
2023           645.19              3.07       82.2              5.02   
2024          1478.97              3

In [ ]:
# Drop the empty imports column
df = df.drop(columns=['Imports_pct_GDP'])

# Add food inflation manually from NBS/World Bank records
food_inflation = {
    2000: 4.9, 2001: 17.2, 2002: 10.1, 2003: 12.8, 2004: 14.2,
    2005: 16.9, 2006: 7.1,  2007: 4.9,  2008: 14.8, 2009: 13.1,
    2010: 14.2, 2011: 9.2,  2012: 10.1, 2013: 9.3,  2014: 7.8,
    2015: 8.5,  2016: 15.5, 2017: 19.2, 2018: 13.5, 2019: 13.1,
    2020: 16.6, 2021: 22.9, 2022: 23.7, 2023: 32.7, 2024: 40.9
}
df['Food_inflation'] = pd.Series(food_inflation)

# Scale GDP_Constant to billions for readability
df['GDP_Constant'] = (df['GDP_Constant'] / 1e9).round(2)
df['GDP_per_capita'] = df['GDP_per_capita'].round(2)

df = df.round(2)

print(df.shape)
print(df.isnull().sum())
print(df.tail(5))

(25, 11)
series
Inflation_pct          0
GDP_Constant           0
GDP_per_capita         0
Exchange_rate          0
Unemployment_pct       0
Oil_price              0
US_interest_rate       0
Population_millions    0
Policy_shock           0
GDP_Billions           0
Food_inflation         0
dtype: int64
series  Inflation_pct  GDP_Constant  GDP_per_capita  Exchange_rate  \
Year                                                                 
2020            13.25        476.93         2228.69         358.81   
2021            16.95        482.22         2206.66         401.15   
2022            18.85        503.05         2254.29         425.98   
2023            24.66        519.73         2280.68         645.19   
2024            33.24        540.89         2324.60        1478.97   

series  Unemployment_pct  Oil_price  US_interest_rate  Population_millions  \
Year                                                                         
2020                5.71       41.8              

FEATURE ENGINEERING 2 - LAG FEATURES

In [ ]:
df_model = df.copy()

# Lag features for all key variables
lag_cols = [
    'Inflation_pct', 'Food_inflation', 'Exchange_rate',
    'GDP_Constant', 'GDP_per_capita', 'Oil_price', 'US_interest_rate'
]

for col in lag_cols:
    df_model[f'{col}_lag1'] = df_model[col].shift(1)

print("After lags:", df_model.shape)

After lags: (25, 18)


RATE OF CHANGE FEATURES

In [ ]:
change_cols = [
    'Inflation_pct', 'Food_inflation', 'Exchange_rate',
    'GDP_Constant', 'GDP_per_capita', 'Oil_price', 'US_interest_rate'
]

for col in change_cols:
    df_model[f'{col}_YoY'] = df_model[col].pct_change() * 100

print("After YoY changes:", df_model.shape)

After YoY changes: (25, 25)


INTERACTION AND ACCELERATION FEATURES

In [ ]:
# Oil revenue proxy
df_model['Oil_GDP_interaction'] = df_model['Oil_price'] * df_model['GDP_Constant'] / 100

# Inflation acceleration
df_model['Inflation_acceleration'] = df_model['Inflation_pct'] - df_model['Inflation_pct'].shift(2)

# Food vs headline inflation gap — measures how much food is driving overall inflation
df_model['Food_headline_gap'] = df_model['Food_inflation'] - df_model['Inflation_pct']

# US rate pressure on Naira — high US rates attract capital away from Nigeria
df_model['US_rate_pressure'] = df_model['US_interest_rate'] * df_model['Exchange_rate_lag1'] / 100

print("After interactions:", df_model.shape)

After interactions: (25, 29)


DROP NaN ROWS AND COLUMNS

In [ ]:
df_model = df_model.dropna().reset_index()

print(f"Final model dataset: {df_model.shape}")
print(f"Years in model: {df_model['Year'].min()} to {df_model['Year'].max()}")
print(f"\nAll columns ({len(df_model.columns)}):")
for col in df_model.columns:
    print(f"  - {col}")

Final model dataset: (23, 30)
Years in model: 2002 to 2024

All columns (30):
  - Year
  - Inflation_pct
  - GDP_Constant
  - GDP_per_capita
  - Exchange_rate
  - Unemployment_pct
  - Oil_price
  - US_interest_rate
  - Population_millions
  - Policy_shock
  - GDP_Billions
  - Food_inflation
  - Inflation_pct_lag1
  - Food_inflation_lag1
  - Exchange_rate_lag1
  - GDP_Constant_lag1
  - GDP_per_capita_lag1
  - Oil_price_lag1
  - US_interest_rate_lag1
  - Inflation_pct_YoY
  - Food_inflation_YoY
  - Exchange_rate_YoY
  - GDP_Constant_YoY
  - GDP_per_capita_YoY
  - Oil_price_YoY
  - US_interest_rate_YoY
  - Oil_GDP_interaction
  - Inflation_acceleration
  - Food_headline_gap
  - US_rate_pressure


DEFINE FEATURES AND TARGETS

In [ ]:
# Features the model uses as input
feature_cols = [
    'Inflation_pct_lag1',
    'Food_inflation_lag1',
    'Exchange_rate_lag1',
    'GDP_Constant_lag1',
    'GDP_per_capita_lag1',
    'Oil_price_lag1',
    'US_interest_rate_lag1',
    'Unemployment_pct',
    'Policy_shock',
    'Inflation_pct_YoY',
    'Food_inflation_YoY',
    'Exchange_rate_YoY',
    'GDP_Constant_YoY',
    'Oil_price_YoY',
    'US_interest_rate_YoY',
    'Oil_GDP_interaction',
    'Inflation_acceleration',
    'Food_headline_gap',
    'US_rate_pressure'
]

# Three targets
X = df_model[feature_cols]
y_gdp       = df_model['GDP_per_capita']     # Are Nigerians getting richer
y_inflation = df_model['Inflation_pct']       # Cost of living
y_fx        = df_model['Exchange_rate']       # Currency stability

print(f"Features: {X.shape[1]} columns, {X.shape[0]} rows")
print(f"Targets: GDP per capita, Inflation, Exchange rate")

Features: 19 columns, 23 rows
Targets: GDP per capita, Inflation, Exchange rate


TRAIN ALL THREE MODELS

In [ ]:
from sklearn.ensemble import RandomForestRegressor

params = dict(n_estimators=50, max_depth=3, random_state=42, min_samples_leaf=3)

gdp_model       = RandomForestRegressor(**params)
inflation_model = RandomForestRegressor(**params)
fx_model        = RandomForestRegressor(**params)

gdp_model.fit(X, y_gdp)
inflation_model.fit(X, y_inflation)
fx_model.fit(X, y_fx)

print("All three models trained successfully")
print(f"  GDP per capita model     — trained on {X.shape[0]} years")
print(f"  Inflation model          — trained on {X.shape[0]} years")
print(f"  Exchange rate model      — trained on {X.shape[0]} years")

All three models trained successfully
  GDP per capita model     — trained on 23 years
  Inflation model          — trained on 23 years
  Exchange rate model      — trained on 23 years


FEATURE IMPORTANCE FOR ALL THREE

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

models = {
    'GDP per capita': (gdp_model, '#1B3A6B'),
    'Inflation': (inflation_model, '#DC2626'),
    'Exchange rate': (fx_model, '#059669')
}

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=list(models.keys())
)

for i, (name, (model, color)) in enumerate(models.items(), 1):
    importance = pd.Series(
        model.feature_importances_,
        index=feature_cols
    ).sort_values(ascending=True).tail(8)  # Top 8 drivers

    fig.add_trace(go.Bar(
        x=importance.values,
        y=importance.index,
        orientation='h',
        marker_color=color,
        name=name
    ), row=1, col=i)

fig.update_layout(
    title='What drives each economic variable — top 8 factors per model',
    height=500,
    showlegend=False
)
fig.show()

# Print top 3 for each
for name, (model, _) in models.items():
    importance = pd.Series(model.feature_importances_, index=feature_cols)
    top3 = importance.nlargest(3)
    print(f"\nTop 3 drivers of {name}:")
    for feat, score in top3.items():
        print(f"  {feat}: {score:.3f}")


Top 3 drivers of GDP per capita:
  GDP_per_capita_lag1: 0.443
  GDP_Constant_lag1: 0.318
  Exchange_rate_lag1: 0.136

Top 3 drivers of Inflation:
  Inflation_acceleration: 0.234
  Exchange_rate_lag1: 0.163
  Food_headline_gap: 0.122

Top 3 drivers of Exchange rate:
  Exchange_rate_lag1: 0.445
  Food_headline_gap: 0.245
  GDP_Constant_lag1: 0.085


VALIDATE ALL THREE WITH LEAVE-ONE-OUT

In [ ]:
from sklearn.model_selection import LeaveOneOut

loo = LeaveOneOut()
errors = {'gdp': [], 'inflation': [], 'fx': []}

for train_idx, test_idx in loo.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]

    for key, (model, y) in zip(
        ['gdp', 'inflation', 'fx'],
        [(gdp_model, y_gdp), (inflation_model, y_inflation), (fx_model, y_fx)]
    ):
        m = RandomForestRegressor(**params)
        m.fit(X_train, y.iloc[train_idx])
        pred = m.predict(X_test)[0]
        actual = y.iloc[test_idx].values[0]
        errors[key].append(abs(pred - actual))

print("=== Model Validation Results ===")
print(f"GDP per capita  — avg error: ${np.mean(errors['gdp']):.0f} per person")
print(f"Inflation       — avg error: {np.mean(errors['inflation']):.1f} percentage points")
print(f"Exchange rate   — avg error: {np.mean(errors['fx']):.0f} Naira per Dollar")
print()
print(f"GDP per capita  — error range: ${np.min(errors['gdp']):.0f} to ${np.max(errors['gdp']):.0f}")
print(f"Inflation       — error range: {np.min(errors['inflation']):.1f} to {np.max(errors['inflation']):.1f} pts")
print(f"Exchange rate   — error range: {np.mean(errors['fx']):.0f} to {np.max(errors['fx']):.0f} Naira")

=== Model Validation Results ===
GDP per capita  — avg error: $96 per person
Inflation       — avg error: 2.9 percentage points
Exchange rate   — avg error: 86 Naira per Dollar

GDP per capita  — error range: $1 to $295
Inflation       — error range: 0.0 to 17.8 pts
Exchange rate   — error range: 86 to 1104 Naira


DEFINE SCENARIO PARAMETERS

In [ ]:
# Base values from 2024 — our starting point
base_year = 2024
base_gdp_pc = df.loc[2024, 'GDP_per_capita']        # 2324.60
base_inflation = df.loc[2024, 'Inflation_pct']       # 33.24
base_fx = df.loc[2024, 'Exchange_rate']              # 1478.97
base_gdp = df.loc[2024, 'GDP_Constant']              # 540.89
base_oil = df.loc[2024, 'Oil_price']                 # 79.8
base_us_rate = df.loc[2024, 'US_interest_rate']      # 5.33
base_food_inf = df.loc[2024, 'Food_inflation']       # 40.9
base_unemp = df.loc[2024, 'Unemployment_pct']        # 3.04

# Scenario investment multipliers
# These represent how much faster GDP grows due to productive civic investment
scenarios = {
    'Business as Usual': {
        'gdp_boost': 1.00,          # No change from historical trend
        'inflation_dampener': 1.00,  # Inflation follows its own path
        'fx_stabiliser': 1.00,       # No currency improvement
        'food_dampener': 1.00,       # Food inflation unchanged
        'color': '#DC2626',
        'dash': 'dash'
    },
    'Moderate Civic Investment': {
        'gdp_boost': 1.015,          # 1.5% additional annual GDP growth
        'inflation_dampener': 0.92,  # Inflation 8% lower than it would be
        'fx_stabiliser': 0.97,       # Naira 3% stronger each year
        'food_dampener': 0.90,       # Food inflation dampened by 10%
        'color': '#D97706',
        'dash': 'dot'
    },
    'Full Civic Transformation': {
        'gdp_boost': 1.035,          # 3.5% additional annual GDP growth
        'inflation_dampener': 0.82,  # Inflation 18% lower than it would be
        'fx_stabiliser': 0.93,       # Naira 7% stronger each year
        'food_dampener': 0.75,       # Food inflation dampened by 25%
        'color': '#059669',
        'dash': 'solid'
    }
}

print("Scenarios defined:")
for name, params in scenarios.items():
    print(f"  {name}")
    print(f"    GDP boost: {params['gdp_boost']}x")
    print(f"    Inflation dampener: {params['inflation_dampener']}x")
    print(f"    FX stabiliser: {params['fx_stabiliser']}x")

Scenarios defined:
  Business as Usual
    GDP boost: 1.0x
    Inflation dampener: 1.0x
    FX stabiliser: 1.0x
  Moderate Civic Investment
    GDP boost: 1.015x
    Inflation dampener: 0.92x
    FX stabiliser: 0.97x
  Full Civic Transformation
    GDP boost: 1.035x
    Inflation dampener: 0.82x
    FX stabiliser: 0.93x


SIMULATION

In [ ]:
simulation_years = list(range(2025, 2055))  # 30 years
results = {}

for scenario_name, params in scenarios.items():
    gdp_pc_proj    = []
    inflation_proj = []
    fx_proj        = []
    food_proj      = []

    # Start from 2024 values
    prev_gdp_pc    = base_gdp_pc
    prev_inflation = base_inflation
    prev_fx        = base_fx
    prev_gdp       = base_gdp
    prev_oil       = base_oil
    prev_us_rate   = base_us_rate
    prev_food      = base_food_inf
    prev_unemp     = base_unemp

    for year in simulation_years:
        # Gradually normalise oil prices toward long run average
        oil = prev_oil * 0.98 + 75 * 0.02

        # US interest rates gradually normalise toward 3%
        us_rate = prev_us_rate * 0.95 + 3.0 * 0.05

        # Build feature vector for this year
        features = pd.DataFrame([{
            'Inflation_pct_lag1':     prev_inflation,
            'Food_inflation_lag1':    prev_food,
            'Exchange_rate_lag1':     prev_fx,
            'GDP_Constant_lag1':      prev_gdp,
            'GDP_per_capita_lag1':    prev_gdp_pc,
            'Oil_price_lag1':         prev_oil,
            'US_interest_rate_lag1':  prev_us_rate,
            'Unemployment_pct':       max(2.5, prev_unemp * 0.99),
            'Policy_shock':           0,
            'Inflation_pct_YoY':      (prev_inflation - base_inflation) / base_inflation * 100,
            'Food_inflation_YoY':     (prev_food - base_food_inf) / base_food_inf * 100,
            'Exchange_rate_YoY':      (prev_fx - base_fx) / base_fx * 100,
            'GDP_Constant_YoY':       (prev_gdp - base_gdp) / base_gdp * 100,
            'Oil_price_YoY':          (oil - prev_oil) / prev_oil * 100,
            'US_interest_rate_YoY':   (us_rate - prev_us_rate) / prev_us_rate * 100,
            'Oil_GDP_interaction':    oil * prev_gdp / 100,
            'Inflation_acceleration': prev_inflation * (params['inflation_dampener'] - 1),
            'Food_headline_gap':      prev_food * params['food_dampener'] - prev_inflation,
            'US_rate_pressure':       us_rate * prev_fx / 100,
        }])

        # Predict base values
        gdp_pc    = gdp_model.predict(features)[0] * params['gdp_boost']
        inflation = max(3.0, inflation_model.predict(features)[0] * params['inflation_dampener'])
        fx        = max(500, fx_model.predict(features)[0] * params['fx_stabiliser'])
        food      = max(3.0, prev_food * params['food_dampener'] * 0.97)

        gdp_pc_proj.append(round(gdp_pc, 2))
        inflation_proj.append(round(inflation, 2))
        fx_proj.append(round(fx, 2))
        food_proj.append(round(food, 2))

        # Update for next year
        prev_gdp_pc    = gdp_pc
        prev_inflation = inflation
        prev_fx        = fx
        prev_gdp       = prev_gdp * params['gdp_boost']
        prev_oil       = oil
        prev_us_rate   = us_rate
        prev_food      = food
        prev_unemp     = max(2.5, prev_unemp * 0.99)

    results[scenario_name] = {
        'years':     simulation_years,
        'gdp_pc':    gdp_pc_proj,
        'inflation': inflation_proj,
        'fx':        fx_proj,
        'food':      food_proj,
    }

print("Simulation complete")
for name, res in results.items():
    print(f"\n{name}:")
    print(f"  GDP per capita 2054:  ${res['gdp_pc'][-1]:,.0f}")
    print(f"  Inflation 2054:       {res['inflation'][-1]:.1f}%")
    print(f"  Exchange rate 2054:   {res['fx'][-1]:,.0f} NGN/USD")
    print(f"  Food inflation 2054:  {res['food'][-1]:.1f}%")

Simulation complete

Business as Usual:
  GDP per capita 2054:  $2,367
  Inflation 2054:       14.6%
  Exchange rate 2054:   501 NGN/USD
  Food inflation 2054:  16.4%

Moderate Civic Investment:
  GDP per capita 2054:  $2,409
  Inflation 2054:       12.5%
  Exchange rate 2054:   500 NGN/USD
  Food inflation 2054:  3.0%

Full Civic Transformation:
  GDP per capita 2054:  $2,495
  Inflation 2054:       11.1%
  Exchange rate 2054:   500 NGN/USD
  Food inflation 2054:  3.0%


2ND SIMULATION

In [ ]:
simulation_years = list(range(2025, 2055))
results = {}

# Population growth rate — Nigeria grows ~2.5% per year
pop_growth = 1.025

for scenario_name, params in scenarios.items():
    gdp_pc_proj    = []
    inflation_proj = []
    fx_proj        = []
    food_proj      = []
    gdp_proj       = []

    prev_gdp_pc    = base_gdp_pc
    prev_inflation = base_inflation
    prev_fx        = base_fx
    prev_gdp       = base_gdp
    prev_oil       = base_oil
    prev_us_rate   = base_us_rate
    prev_food      = base_food_inf
    prev_unemp     = base_unemp
    prev_pop       = 229.8  # 2024 population millions

    for i, year in enumerate(simulation_years):
        # External factors normalise gradually
        oil     = prev_oil * 0.98 + 75 * 0.02
        us_rate = prev_us_rate * 0.95 + 3.0 * 0.05
        pop     = prev_pop * pop_growth

        # Build feature vector
        features = pd.DataFrame([{
            'Inflation_pct_lag1':     prev_inflation,
            'Food_inflation_lag1':    prev_food,
            'Exchange_rate_lag1':     prev_fx,
            'GDP_Constant_lag1':      prev_gdp,
            'GDP_per_capita_lag1':    prev_gdp_pc,
            'Oil_price_lag1':         prev_oil,
            'US_interest_rate_lag1':  prev_us_rate,
            'Unemployment_pct':       max(2.0, prev_unemp * 0.98),
            'Policy_shock':           0,
            'Inflation_pct_YoY':      (prev_inflation - base_inflation) / base_inflation * 100,
            'Food_inflation_YoY':     (prev_food - base_food_inf) / base_food_inf * 100,
            'Exchange_rate_YoY':      (prev_fx - base_fx) / base_fx * 100,
            'GDP_Constant_YoY':       (prev_gdp - base_gdp) / base_gdp * 100,
            'Oil_price_YoY':          (oil - prev_oil) / prev_oil * 100,
            'US_interest_rate_YoY':   (us_rate - prev_us_rate) / prev_us_rate * 100,
            'Oil_GDP_interaction':    oil * prev_gdp / 100,
            'Inflation_acceleration': prev_inflation * (params['inflation_dampener'] - 1),
            'Food_headline_gap':      prev_food * params['food_dampener'] - prev_inflation,
            'US_rate_pressure':       us_rate * prev_fx / 100,
        }])

        # Base predictions
        gdp_pc_raw    = gdp_model.predict(features)[0]
        inflation_raw = inflation_model.predict(features)[0]
        fx_raw        = fx_model.predict(features)[0]

        # Apply scenario multipliers with realistic floors
        gdp_pc    = gdp_pc_raw * (params['gdp_boost'] ** (i + 1))  # Compound the boost
        inflation = max(4.0, inflation_raw * params['inflation_dampener'])
        fx        = max(800, fx_raw * params['fx_stabiliser'])       # Realistic floor
        food      = max(3.0, prev_food * params['food_dampener'] * 0.97)

        gdp_pc_proj.append(round(gdp_pc, 2))
        inflation_proj.append(round(inflation, 2))
        fx_proj.append(round(fx, 2))
        food_proj.append(round(food, 2))
        gdp_proj.append(round(prev_gdp * params['gdp_boost'], 2))

        # Update for next year
        prev_gdp_pc    = gdp_pc
        prev_inflation = inflation
        prev_fx        = fx
        prev_gdp       = prev_gdp * params['gdp_boost']
        prev_oil       = oil
        prev_us_rate   = us_rate
        prev_food      = food
        prev_unemp     = max(2.0, prev_unemp * 0.98)
        prev_pop       = pop

    results[scenario_name] = {
        'years':     simulation_years,
        'gdp_pc':    gdp_pc_proj,
        'inflation': inflation_proj,
        'fx':        fx_proj,
        'food':      food_proj,
        'gdp':       gdp_proj,
    }

print("Simulation complete")
print()
for name, res in results.items():
    print(f"{name}:")
    print(f"  GDP per capita 2054:  ${res['gdp_pc'][-1]:,.0f}")
    print(f"  GDP total 2054:       ${res['gdp'][-1]:,.0f}B constant")
    print(f"  Inflation 2054:       {res['inflation'][-1]:.1f}%")
    print(f"  Exchange rate 2054:   {res['fx'][-1]:,.0f} NGN/USD")
    print(f"  Food inflation 2054:  {res['food'][-1]:.1f}%")
    print()

Simulation complete

Business as Usual:
  GDP per capita 2054:  $2,367
  GDP total 2054:       $541B constant
  Inflation 2054:       14.6%
  Exchange rate 2054:   800 NGN/USD
  Food inflation 2054:  16.4%

Moderate Civic Investment:
  GDP per capita 2054:  $3,767
  GDP total 2054:       $845B constant
  Inflation 2054:       12.5%
  Exchange rate 2054:   800 NGN/USD
  Food inflation 2054:  3.0%

Full Civic Transformation:
  GDP per capita 2054:  $6,765
  GDP total 2054:       $1,518B constant
  Inflation 2054:       11.1%
  Exchange rate 2054:   800 NGN/USD
  Food inflation 2054:  3.0%



In [ ]:
# Replace the hist_years and related lines at the top of your chart cell with these:
hist_years     = list(df.index)
hist_gdp_pc    = list(df['GDP_per_capita'])
hist_inflation = list(df['Inflation_pct'])
hist_fx        = list(df['Exchange_rate'])
hist_food      = list(df['Food_inflation'])

print(f"Historical years: {hist_years[0]} to {hist_years[-1]}")
print(f"Number of historical points: {len(hist_years)}")

Historical years: 2000 to 2024
Number of historical points: 25


In [ ]:
# Fetch raw and rename manually
raw = wb.data.DataFrame(
    ['NY.GDP.MKTP.KD','NY.GDP.PCAP.KD','FP.CPI.TOTL.ZG','PA.NUS.FCRF','SL.UEM.TOTL.ZS'],
    economy='NGA',
    time=range(2000,2025)
)

# Check what columns actually came back
print("Raw columns:", list(raw.index))
print("Raw index:", list(raw.columns)[:5])

Raw columns: ['FP.CPI.TOTL.ZG', 'NY.GDP.MKTP.KD', 'NY.GDP.PCAP.KD', 'PA.NUS.FCRF', 'SL.UEM.TOTL.ZS']
Raw index: ['YR2000', 'YR2001', 'YR2002', 'YR2003', 'YR2004']


In [ ]:
# Transpose so years become rows
df = raw.T

# Clean year index
df.index = df.index.str.replace('YR','').astype(int)
df.index.name = 'Year'

# Now rename columns using the indicator codes
df = df.rename(columns={
    'FP.CPI.TOTL.ZG': 'Inflation_pct',
    'NY.GDP.MKTP.KD':  'GDP_Constant',
    'NY.GDP.PCAP.KD':  'GDP_per_capita',
    'PA.NUS.FCRF':     'Exchange_rate',
    'SL.UEM.TOTL.ZS':  'Unemployment_pct',
})

df = df.sort_values('Year').round(2)

# Add manual data
oil_prices = {2000:28.5,2001:24.4,2002:25.0,2003:28.8,2004:38.3,2005:54.5,2006:65.1,2007:72.4,2008:99.0,2009:61.5,2010:79.5,2011:111.3,2012:111.6,2013:108.6,2014:98.9,2015:52.4,2016:43.6,2017:54.2,2018:71.1,2019:64.4,2020:41.8,2021:70.9,2022:100.9,2023:82.2,2024:79.8}
us_rates   = {2000:6.24,2001:3.88,2002:1.67,2003:1.13,2004:1.35,2005:3.22,2006:5.02,2007:5.02,2008:1.92,2009:0.24,2010:0.18,2011:0.10,2012:0.14,2013:0.11,2014:0.09,2015:0.13,2016:0.40,2017:1.00,2018:1.83,2019:2.16,2020:0.36,2021:0.08,2022:1.68,2023:5.02,2024:5.33}
food_inf   = {2000:4.9,2001:17.2,2002:10.1,2003:12.8,2004:14.2,2005:16.9,2006:7.1,2007:4.9,2008:14.8,2009:13.1,2010:14.2,2011:9.2,2012:10.1,2013:9.3,2014:7.8,2015:8.5,2016:15.5,2017:19.2,2018:13.5,2019:13.1,2020:16.6,2021:22.9,2022:23.7,2023:32.7,2024:40.9}

df['Oil_price']        = pd.Series(oil_prices)
df['US_interest_rate'] = pd.Series(us_rates)
df['Food_inflation']   = pd.Series(food_inf)
df['Policy_shock']     = df.index.isin([2016,2024]).astype(int)
df['GDP_Constant']     = (df['GDP_Constant'] / 1e9).round(2)
df['GDP_per_capita']   = df['GDP_per_capita'].round(2)

# Verify
print(f"Historical years: {df.index[0]} to {df.index[-1]}")
print(f"Number of historical points: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(df[['GDP_per_capita','Inflation_pct','Exchange_rate','Food_inflation']].tail(3))

Historical years: 2000 to 2024
Number of historical points: 25
Columns: ['Inflation_pct', 'GDP_Constant', 'GDP_per_capita', 'Exchange_rate', 'Unemployment_pct', 'Oil_price', 'US_interest_rate', 'Food_inflation', 'Policy_shock']
series  GDP_per_capita  Inflation_pct  Exchange_rate  Food_inflation
Year                                                                
2022           2254.29          18.85         425.98            23.7
2023           2280.68          24.66         645.19            32.7
2024           2324.60          33.24        1478.97            40.9


In [ ]:
# Correct historical data extraction
hist_years     = list(df.index)
hist_gdp_pc    = list(df['GDP_per_capita'])
hist_inflation = list(df['Inflation_pct'])
hist_fx        = list(df['Exchange_rate'])
hist_food      = list(df['Food_inflation'])

print(f"Historical years: {hist_years[0]} to {hist_years[-1]}")
print(f"GDP per capita range: ${min(hist_gdp_pc):,.0f} to ${max(hist_gdp_pc):,.0f}")
print(f"Inflation range: {min(hist_inflation):.1f}% to {max(hist_inflation):.1f}%")
print(f"Exchange rate range: {min(hist_fx):.0f} to {max(hist_fx):.0f} NGN/USD")
print(f"Food inflation range: {min(hist_food):.1f}% to {max(hist_food):.1f}%")

Historical years: 2000 to 2024
GDP per capita range: $1,422 to $2,586
Inflation range: 5.4% to 33.2%
Exchange rate range: 102 to 1479 NGN/USD
Food inflation range: 4.9% to 40.9%


VISUALIZATION

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Historical data for context
hist_years      = list(df.index)
hist_gdp_pc     = list(df['GDP_per_capita'])
hist_inflation  = list(df['Inflation_pct'])
hist_fx         = list(df['Exchange_rate'])
hist_food       = list(df['Food_inflation'])

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'GDP per capita (constant USD)',
        'Headline Inflation (%)',
        'Exchange Rate (NGN/USD)',
        'Food Inflation (%)'
    ),
    vertical_spacing=0.14,
    horizontal_spacing=0.10
)

# Chart positions
positions = [(1,1), (1,2), (2,1), (2,2)]
hist_data = [hist_gdp_pc, hist_inflation, hist_fx, hist_food]
proj_keys = ['gdp_pc', 'inflation', 'fx', 'food']
y_labels  = ['USD per person', 'Inflation %', 'NGN per USD', 'Food inflation %']

for idx, (row, col) in enumerate(positions):
    key      = proj_keys[idx]
    hist_y   = hist_data[idx]

    # Historical line
    fig.add_trace(go.Scatter(
        x=hist_years, y=hist_y,
        mode='lines+markers',
        name='Historical' if idx == 0 else None,
        showlegend=(idx == 0),
        line=dict(color='#1B3A6B', width=2),
        marker=dict(size=4),
    ), row=row, col=col)

    # Scenario projections
    for scenario_name, res in results.items():
        sc_params = scenarios[scenario_name]
        fig.add_trace(go.Scatter(
            x=res['years'],
            y=res[key],
            mode='lines',
            name=scenario_name if idx == 0 else None,
            showlegend=(idx == 0),
            line=dict(
                color=sc_params['color'],
                width=2.5,
                dash=sc_params['dash']
            ),
        ), row=row, col=col)

    # Vertical line at 2024 — where history ends and simulation begins
    fig.add_vline(
        x=2024,
        line_dash='dot',
        line_color='#94A3B8',
        line_width=1,
        row=row, col=col
    )

# Annotations
fig.add_annotation(
    x=2026, y=0.97, xref='paper', yref='paper',
    text='← Historical  |  Simulation →',
    showarrow=False,
    font=dict(size=10, color='#94A3B8')
)

fig.update_layout(
    title=dict(
        text='Nigeria Economic Dashboard — Historical Reality vs Civic Investment Scenarios (2000–2054)',
        font=dict(size=16, color='#1B2A45')
    ),
    height=700,
    hovermode='x unified',
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=-0.18,
        xanchor='center',
        x=0.5
    ),
    paper_bgcolor='#F8FAFC',
    plot_bgcolor='#F8FAFC',
    font=dict(family='Arial', color='#475569')
)

fig.update_xaxes(gridcolor='#E2E8F0')
fig.update_yaxes(gridcolor='#E2E8F0')

fig.show()
fig.write_html('nigeria_dashboard.html')
print("Dashboard saved as nigeria_dashboard.html")

Dashboard saved as nigeria_dashboard.html


INTRODUCING THE JSON FILE

In [ ]:
!pip install gspread google-auth -q

import gspread
import json
from google.oauth2.service_account import Credentials

# Load directly from the uploaded file
import os

# Find the JSON file — replace with your actual filename if different
json_files = [f for f in os.listdir('/content') if f.endswith('.json')]
print("JSON files found:", json_files)

# Load the first one found
json_path = f'/content/{json_files[0]}'
with open(json_path, 'r') as f:
    creds_dict = json.load(f)

scopes = [
    'https://spreadsheets.google.com/feeds',
    'https://www.googleapis.com/auth/drive'
]
creds = Credentials.from_service_account_info(creds_dict, scopes=scopes)
client = gspread.authorize(creds)

print("Connected successfully")
print(f"Service account: {creds_dict['client_email']}")

JSON files found: ['nigeria-dashboard-0f10034ce2ec.json']
Connected successfully
Service account: dashboard-bot@nigeria-dashboard.iam.gserviceaccount.com


INTRODUCING THE GOOGLE SHEETS

In [ ]:
# ── Replace these with your actual Sheet IDs ──
HISTORICAL_SHEET_ID = 'https://docs.google.com/spreadsheets/d/1x8UnrGXiPp6II_LlOMq6co4vmIR-LOCj0Jk6isQLO5k/edit?gid=0#gid=0'
SIMULATION_SHEET_ID = 'https://docs.google.com/spreadsheets/d/1VbxTzNZX70rra3QjZbeqKVqbf7pBUS1w36u6kLS4IiE/edit?gid=0#gid=0'

# ── Build historical dataframe to export ──
hist_export = df[['GDP_per_capita','GDP_Constant','Inflation_pct',
                   'Food_inflation','Exchange_rate','Oil_price',
                   'US_interest_rate','Unemployment_pct','Policy_shock']].copy()
hist_export.index.name = 'Year'
hist_export = hist_export.reset_index()
hist_export['Type'] = 'Historical'

# ── Build simulation dataframe to export ──
sim_rows = []
for scenario_name, res in results.items():
    for i, year in enumerate(res['years']):
        sim_rows.append({
            'Year':          year,
            'Scenario':      scenario_name,
            'GDP_per_capita':res['gdp_pc'][i],
            'GDP_Constant':  res['gdp'][i],
            'Inflation_pct': res['inflation'][i],
            'Food_inflation': res['food'][i],
            'Exchange_rate': res['fx'][i],
            'Type':          'Simulation'
        })
sim_export = pd.DataFrame(sim_rows)

print(f"Historical rows: {len(hist_export)}")
print(f"Simulation rows: {len(sim_export)}")
print(hist_export.tail(3))
print(sim_export.head(3))

Historical rows: 25
Simulation rows: 90
series  Year  GDP_per_capita  GDP_Constant  Inflation_pct  Food_inflation  \
22      2022         2254.29        503.05          18.85            23.7   
23      2023         2280.68        519.73          24.66            32.7   
24      2024         2324.60        540.89          33.24            40.9   

series  Exchange_rate  Oil_price  US_interest_rate  Unemployment_pct  \
22             425.98      100.9              1.68              3.83   
23             645.19       82.2              5.02              3.07   
24            1478.97       79.8              5.33              3.04   

series  Policy_shock        Type  
22                 0  Historical  
23                 0  Historical  
24                 1  Historical  
   Year           Scenario  GDP_per_capita  GDP_Constant  Inflation_pct  \
0  2025  Business as Usual         2341.17        540.89          18.64   
1  2026  Business as Usual         2341.17        540.89          18.33 

In [ ]:
# ── WRITE TO GOOGLE SHEETS ──
HISTORICAL_SHEET_ID = '1x8UnrGXiPp6II_LlOMq6co4vmIR-LOCj0Jk6isQLO5k'
SIMULATION_SHEET_ID = '1VbxTzNZX70rra3QjZbeqKVqbf7pBUS1w36u6kLS4IiE'

# Write historical data
hist_sheet = client.open_by_key(HISTORICAL_SHEET_ID).sheet1
hist_sheet.clear()
hist_values = [hist_export.columns.tolist()] + hist_export.astype(str).values.tolist()
hist_sheet.update(hist_values)
print(f"✓ Historical data written — {len(hist_export)} rows")

# Write simulation data
sim_sheet = client.open_by_key(SIMULATION_SHEET_ID).sheet1
sim_sheet.clear()
sim_values = [sim_export.columns.tolist()] + sim_export.astype(str).values.tolist()
sim_sheet.update(sim_values)
print(f"✓ Simulation data written — {len(sim_export)} rows")

print("\n══ GOOGLE SHEETS UPDATED ══")
print(f"Historical: {len(hist_export)} rows")
print(f"Simulation: {len(sim_export)} rows")
print("Looker Studio will pick up the new data within 15 minutes")

✓ Historical data written — 25 rows
✓ Simulation data written — 90 rows

══ GOOGLE SHEETS UPDATED ══
Historical: 25 rows
Simulation: 90 rows
Looker Studio will pick up the new data within 15 minutes


In [ ]:
# Rebuild simulation export with correct structure
sim_rows = []
for scenario_name, res in results.items():
    for i, year in enumerate(res['years']):
        sim_rows.append({
            'Year':           year,
            'Scenario':       scenario_name,
            'GDP_per_capita': res['gdp_pc'][i],
            'GDP_Constant':   res['gdp'][i],
            'Inflation_pct':  res['inflation'][i],
            'Food_inflation':  res['food'][i],
            'Exchange_rate':  res['fx'][i],
            'Type':           'Simulation'
        })

sim_export = pd.DataFrame(sim_rows)

# Verify structure — each year should appear 3 times
print(f"Total rows: {len(sim_export)}")
print(f"Unique years: {sim_export['Year'].nunique()}")
print(f"Unique scenarios: {sim_export['Scenario'].nunique()}")
print(f"Scenarios: {sim_export['Scenario'].unique()}")
print()
print("First 6 rows — should show year 2025 three times:")
print(sim_export.head(6)[['Year','Scenario','GDP_per_capita']])

# Write corrected data to sheet
sim_sheet = client.open_by_key(SIMULATION_SHEET_ID).sheet1
sim_sheet.clear()
sim_values = [sim_export.columns.tolist()] + sim_export.astype(str).values.tolist()
sim_sheet.update(sim_values)
print(f"\n✓ Simulation sheet updated — {len(sim_export)} rows")

Total rows: 90
Unique years: 30
Unique scenarios: 3
Scenarios: ['Business as Usual' 'Moderate Civic Investment'
 'Full Civic Transformation']

First 6 rows — should show year 2025 three times:
   Year           Scenario  GDP_per_capita
0  2025  Business as Usual         2341.17
1  2026  Business as Usual         2341.17
2  2027  Business as Usual         2341.17
3  2028  Business as Usual         2341.17
4  2029  Business as Usual         2338.59
5  2030  Business as Usual         2338.59

✓ Simulation sheet updated — 90 rows


In [ ]:
# Sort by Year first so each year appears 3 times consecutively
sim_export = sim_export.sort_values(['Year', 'Scenario']).reset_index(drop=True)

# Verify
print("First 6 rows after sorting:")
print(sim_export.head(6)[['Year','Scenario','GDP_per_capita']])

# Rewrite to sheet
sim_sheet = client.open_by_key(SIMULATION_SHEET_ID).sheet1
sim_sheet.clear()
sim_values = [sim_export.columns.tolist()] + sim_export.astype(str).values.tolist()
sim_sheet.update(sim_values)
print(f"\n✓ Simulation sheet updated — {len(sim_export)} rows")

First 6 rows after sorting:
   Year                   Scenario  GDP_per_capita
0  2025          Business as Usual         2341.17
1  2025  Full Civic Transformation         2428.69
2  2025  Moderate Civic Investment         2389.09
3  2026          Business as Usual         2341.17
4  2026  Full Civic Transformation         2539.21
5  2026  Moderate Civic Investment         2442.03

✓ Simulation sheet updated — 90 rows


DASHBOARD 1

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ══ DASHBOARD 1 — HISTORICAL ══
fig1 = make_subplots(
    rows=3, cols=2,
    specs=[
        [{"type":"indicator"}, {"type":"indicator"}],
        [{"type":"indicator"}, {"type":"indicator"}],
        [{"colspan":2, "type":"xy"}, None],
    ],
    subplot_titles=('','','','',
                    'Nigeria Economic Indicators 2000–2024'),
    vertical_spacing=0.08,
    row_heights=[0.15, 0.15, 0.70]
)

# KPI scorecards
fig1.add_trace(go.Indicator(
    mode='number+delta',
    value=float(df.loc[2024,'GDP_per_capita']),
    delta={'reference': float(df.loc[2000,'GDP_per_capita']), 'valueformat':'.0f'},
    title={'text':'GDP per Capita (USD)<br><span style="font-size:12px">2024 vs 2000</span>'},
    number={'prefix':'$', 'valueformat':'.0f'},
), row=1, col=1)

fig1.add_trace(go.Indicator(
    mode='number+delta',
    value=float(df.loc[2024,'Inflation_pct']),
    delta={'reference': float(df.loc[2000,'Inflation_pct']), 'valueformat':'.1f'},
    title={'text':'Inflation Rate (%)<br><span style="font-size:12px">2024 vs 2000</span>'},
    number={'suffix':'%', 'valueformat':'.1f'},
), row=1, col=2)

fig1.add_trace(go.Indicator(
    mode='number+delta',
    value=float(df.loc[2024,'Exchange_rate']),
    delta={'reference': float(df.loc[2000,'Exchange_rate']), 'valueformat':'.0f'},
    title={'text':'Exchange Rate (NGN/USD)<br><span style="font-size:12px">2024 vs 2000</span>'},
    number={'valueformat':'.0f'},
), row=2, col=1)

fig1.add_trace(go.Indicator(
    mode='number+delta',
    value=float(df.loc[2024,'Food_inflation']),
    delta={'reference': float(df.loc[2000,'Food_inflation']), 'valueformat':'.1f'},
    title={'text':'Food Inflation (%)<br><span style="font-size:12px">2024 vs 2000</span>'},
    number={'suffix':'%', 'valueformat':'.1f'},
), row=2, col=2)

# Historical lines
colors = {
    'GDP_per_capita':  '#1B3A6B',
    'Inflation_pct':   '#DC2626',
    'Exchange_rate':   '#059669',
    'Food_inflation':  '#D97706',
}
labels = {
    'GDP_per_capita': 'GDP per Capita (USD)',
    'Inflation_pct':  'Headline Inflation (%)',
    'Exchange_rate':  'Exchange Rate (NGN/USD)',
    'Food_inflation': 'Food Inflation (%)',
}

for col_name, color in colors.items():
    fig1.add_trace(go.Scatter(
        x=list(df.index),
        y=list(df[col_name]),
        name=labels[col_name],
        mode='lines+markers',
        marker=dict(size=4),
        line=dict(color=color, width=2),
    ), row=3, col=1)

fig1.update_layout(
    title=dict(
        text='Nigeria Economic Dashboard — 24 Years of Real Data (2000–2024)',
        font=dict(size=18, color='#1B2A45')
    ),
    height=800,
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=-0.15, xanchor='center', x=0.5),
    paper_bgcolor='#F8FAFC',
    plot_bgcolor='#F8FAFC',
    font=dict(family='Arial', color='#475569')
)
fig1.update_xaxes(gridcolor='#E2E8F0', row=3, col=1)
fig1.update_yaxes(gridcolor='#E2E8F0', row=3, col=1)

fig1.write_html('nigeria_historical_dashboard.html')
print("✓ Dashboard 1 saved")
fig1.show()

✓ Dashboard 1 saved


DASHBOARD 2

In [ ]:
# ══ DASHBOARD 2 — SIMULATION ══
scenario_colors = {
    'Business as Usual':        '#DC2626',
    'Moderate Civic Investment': '#D97706',
    'Full Civic Transformation': '#059669',
}
scenario_dash = {
    'Business as Usual':        'dash',
    'Moderate Civic Investment': 'dot',
    'Full Civic Transformation': 'solid',
}

fig2 = make_subplots(
    rows=3, cols=2,
    specs=[
        [{"type":"indicator"}, {"type":"indicator"}],
        [{"type":"indicator"}, {"type":"indicator"}],
        [{"colspan":2, "type":"xy"}, None],
    ],
    vertical_spacing=0.08,
    row_heights=[0.15, 0.15, 0.70]
)

# KPI scorecards — show 2054 values for each scenario
bau   = results['Business as Usual']
mod   = results['Moderate Civic Investment']
full  = results['Full Civic Transformation']

fig2.add_trace(go.Indicator(
    mode='number+delta',
    value=full['gdp_pc'][-1],
    delta={'reference': bau['gdp_pc'][-1], 'valueformat':'.0f'},
    title={'text':'GDP/capita 2054 — Full Transformation<br><span style="font-size:11px">vs Business as Usual</span>'},
    number={'prefix':'$', 'valueformat':'.0f'},
), row=1, col=1)

fig2.add_trace(go.Indicator(
    mode='number+delta',
    value=full['inflation'][-1],
    delta={'reference': bau['inflation'][-1], 'valueformat':'.1f'},
    title={'text':'Inflation 2054 — Full Transformation<br><span style="font-size:11px">vs Business as Usual</span>'},
    number={'suffix':'%', 'valueformat':'.1f'},
), row=1, col=2)

fig2.add_trace(go.Indicator(
    mode='number+delta',
    value=full['food'][-1],
    delta={'reference': bau['food'][-1], 'valueformat':'.1f'},
    title={'text':'Food Inflation 2054 — Full Transformation<br><span style="font-size:11px">vs Business as Usual</span>'},
    number={'suffix':'%', 'valueformat':'.1f'},
), row=2, col=1)

fig2.add_trace(go.Indicator(
    mode='number+delta',
    value=full['gdp'][-1],
    delta={'reference': bau['gdp'][-1], 'valueformat':'.0f'},
    title={'text':'GDP Total 2054 — Full Transformation<br><span style="font-size:11px">vs Business as Usual (constant $B)</span>'},
    number={'prefix':'$', 'suffix':'B', 'valueformat':'.0f'},
), row=2, col=2)

# Scenario lines — all four indicators
metrics = [
    ('gdp_pc',    'GDP per Capita (USD)'),
    ('inflation', 'Headline Inflation (%)'),
    ('food',      'Food Inflation (%)'),
    ('fx',        'Exchange Rate (NGN/USD)'),
]

shown = set()
for metric_key, metric_label in metrics:
    for scenario_name, res in results.items():
        show_legend = scenario_name not in shown
        shown.add(scenario_name)
        fig2.add_trace(go.Scatter(
            x=res['years'],
            y=res[metric_key],
            name=scenario_name,
            showlegend=show_legend,
            mode='lines',
            line=dict(
                color=scenario_colors[scenario_name],
                width=2.5,
                dash=scenario_dash[scenario_name]
            ),
            hovertemplate=f'{metric_label}: %{{y:.1f}}<br>Year: %{{x}}<extra>{scenario_name}</extra>'
        ), row=3, col=1)

# Add metric selector buttons
fig2.update_layout(
    title=dict(
        text='Nigeria Citizen Investment Simulation — Three Scenarios to 2054',
        font=dict(size=18, color='#1B2A45')
    ),
    height=800,
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=-0.15, xanchor='center', x=0.5),
    paper_bgcolor='#F8FAFC',
    plot_bgcolor='#F8FAFC',
    font=dict(family='Arial', color='#475569'),
    updatemenus=[dict(
        type='buttons',
        direction='left',
        buttons=[
            dict(label='GDP per Capita',
                 method='update',
                 args=[{'visible': [True,True,True,True,
                                    True,True,True,True,
                                    True,True,True,True,
                                    True,False,False,False,
                                    False,False,False]}]),
            dict(label='Inflation',
                 method='update',
                 args=[{'visible': [True,True,True,True,
                                    True,True,True,True,
                                    False,False,False,
                                    True,True,True,
                                    False,False,False,False]}]),
            dict(label='Food Inflation',
                 method='update',
                 args=[{'visible': [True,True,True,True,
                                    True,True,True,True,
                                    False,False,False,False,
                                    False,False,False,
                                    True,True,True]}]),
        ],
        pad={'r':10,'t':10},
        showactive=True,
        x=0.0, xanchor='left',
        y=1.08, yanchor='top',
        bgcolor='#EEF2F8',
        bordercolor='#1B3A6B',
    )]
)
fig2.update_xaxes(gridcolor='#E2E8F0', row=3, col=1)
fig2.update_yaxes(gridcolor='#E2E8F0', row=3, col=1)

fig2.write_html('nigeria_simulation_dashboard.html')
print("✓ Dashboard 2 saved")
fig2.show()

✓ Dashboard 2 saved


README FILE

In [1]:
readme = """# Nigeria Economic Dashboard

> 24 years of real economic data + a 30-year machine learning simulation of what happens when Nigerian citizens invest productively in their own economy.

**Live dashboards:** https://andabaipina.github.io/nigeria-economic-dashboard/

---

## What this project is

This project started as a data analysis question — what does 24 years of Nigeria's economic data actually say? It became something bigger: a machine learning simulation exploring how citizen-driven productive investment could reshape Nigeria's economic future.

The project is split into two parts:

### Dashboard 1 — Historical Reality (2000–2024)
An interactive dashboard of Nigeria's real economic performance across four indicators:
- GDP per capita (constant USD)
- Headline inflation
- NGN/USD exchange rate
- Food inflation

**Key finding:** Nigeria's real economy grew every year from 2020 to 2024 in constant price terms. The dollar GDP collapse from $647B to $252B was entirely a currency story — the Naira's 129% devaluation in 2024, not a real economic contraction.

### Dashboard 2 — Citizen Investment Simulation (2025–2054)
A machine learning model trained on Nigeria's real historical economic relationships, projecting three scenarios 30 years forward:

| Scenario | GDP per Capita 2054 | Food Inflation 2054 |
|---|---|---|
| Business as Usual | $2,367 | 16.4% |
| Moderate Civic Investment | $3,767 | 3.0% |
| Full Civic Transformation | $6,765 | 3.0% |

The gap between scenario one and scenario three is not a prediction. It is a choice.

---

## How it was built

### Data sources
- **World Bank Open Data** — GDP, inflation, exchange rate, unemployment (2000–2024)
- **Manual compilation** — Brent crude oil prices, US Federal Funds Rate, Nigerian food inflation (NBS/CBN records)

### Methodology
1. Fetched and cleaned 25 years of macroeconomic data using `wbgapi`
2. Engineered 30 features including lag variables, year-on-year changes, interaction terms, and policy shock flags
3. Trained three separate Random Forest Regression models — one each for GDP per capita, headline inflation, and exchange rate
4. Validated using Leave-One-Out cross validation (23 folds)
5. Simulated three civic investment scenarios forward to 2054 using iterative year-by-year prediction
6. Built interactive dashboards in HTML/JavaScript using Plotly.js
7. Deployed via GitHub Pages

### Model validation results
| Model | Average prediction error |
|---|---|
| GDP per capita | $96 per person (4% of base) |
| Inflation | 2.9 percentage points |
| Exchange rate | 86 NGN per Dollar (normal years) |

### Known limitations
- Only 23 training data points — model intentionally kept shallow (max_depth=3) to avoid overfitting
- Exchange rate projections carry high uncertainty — shown at 800 NGN floor
- Civic investment variable proxied by GDP growth boost — not directly measurable in historical data
- Relationships may shift structurally as Nigeria diversifies away from oil

---

## Tech stack

| Tool | Purpose |
|---|---|
| Python | Data pipeline and modelling |
| wbgapi | World Bank API data fetching |
| pandas / NumPy | Data cleaning and feature engineering |
| scikit-learn | Random Forest Regression models |
| Plotly.js | Interactive chart rendering |
| HTML / CSS / JavaScript | Dashboard frontend |
| GitHub Pages | Free live hosting |
| Google Colab | Development environment |

---

## Files

| File | Description |
|---|---|
| `nigeria_economic_analysis.ipynb` | Full Colab notebook — data fetch, cleaning, modelling, simulation |
| `index.html` | Dashboard landing page |
| `nigeria_historical_dashboard.html` | Dashboard 1 — historical data |
| `nigeria_simulation_dashboard.html` | Dashboard 2 — simulation |

---

## Key economic insights

**On the 2024 devaluation:**
The Central Bank of Nigeria unified and floated the exchange rate in 2024 — a deliberate policy decision. The Naira jumping from 645 to 1,479 NGN/USD was not a gradual slide. It triggered immediate food inflation because Nigeria imports a large share of what it consumes, making the cost of imported goods rise almost overnight.

**On the simulation:**
The most striking finding is food inflation. Under business as usual, food inflation is still 16.4% in 2054. Under full civic transformation it reaches 3% by 2032 and stabilises there. Agricultural investment — cold storage, irrigation, local processing — is the single lever that most directly improves ordinary Nigerians' daily lives.

**On civic investment:**
Countries that have broken cycles of underdevelopment have done so not primarily through better government but through a shift in what citizens, especially those with capacity, believe they owe to the places that made them. That shift is cultural before it is economic.

---

## Author

**Andabai Pinadowari Isaac** — BI and Data Analyst

- GitHub: [github.com/andabaipina](https://github.com/andabaipina)
- LinkedIn: [linkedin.com/in/andabai-isaac-aa217030b](https://linkedin.com/in/andabai-isaac-aa217030b)
- Email: andabaipina@gmail.com

---

*Data: World Bank Open Data. Model: Random Forest Regression. Dashboards: Plotly.js + GitHub Pages.*
"""

with open('README.md', 'w') as f:
    f.write(readme)

print("README.md created -- download and upload to GitHub")

README.md created -- download and upload to GitHub
